<a href="https://colab.research.google.com/github/Luyao-Xu/3-Class-Speech-Command-Classification/blob/main/03_DL_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

base_path = '/content/drive/MyDrive/speech_commands_project'
log_path = '/content/drive/MyDrive/speech_commands_project/logs'
results_path = '/content/drive/MyDrive/speech_commands_project/results'

os.makedirs(log_path, exist_ok=True)
os.makedirs(results_path, exist_ok=True)

print(f"Directories ready.\nLogs: {log_path}\nResults: {results_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Directories ready.
Logs: /content/drive/MyDrive/speech_commands_project/logs
Results: /content/drive/MyDrive/speech_commands_project/results


### Data Preparation (The 2D Loader)


In [ ]:
!pip install "numpy<2.0"
!pip install -q tf_keras
!pip install -q tensorflow-model-optimization
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import pandas as pd
import time
import numpy as np
import tensorflow as tf
import tf_keras
from tf_keras import layers, models
from tf_keras.optimizers import Adam
import tensorflow_model_optimization as tfmot


In [ ]:
# Description: Comparison of Standard CNN, SqueezeNet, and UltraLight with QAT/PTQ
# Author: [Xu Luyao]

config = {
    "version": "1.2.0",
    "input_shape": (40, 32, 1),
    "num_classes": 3,
    "batch_size": 32,
    "initial_epochs": 30,
    "qat_fine_tune_epochs": 8,
    "learning_rate_base": 0.001,
    "learning_rate_qat": 1e-5,
    "random_seed": 42
}

import numpy as np
import tensorflow as tf
tf.random.set_seed(config["random_seed"])
np.random.seed(config["random_seed"])

In [ ]:
# Path to your processed data
processed_path = '/content/drive/MyDrive/speech_commands_project/data/processed'

# Load files
X_train = np.load(os.path.join(processed_path, 'X_train.npy'))
y_train = np.load(os.path.join(processed_path, 'y_train.npy'))
X_val = np.load(os.path.join(processed_path, 'X_val.npy'))
y_val = np.load(os.path.join(processed_path, 'y_val.npy'))
X_test = np.load(os.path.join(processed_path, 'X_test.npy'))
y_test = np.load(os.path.join(processed_path, 'y_test.npy'))

# Reshape for CNN: (Samples, Height, Width, Channels)
# Your MFCCs are 40x32
X_train = X_train.reshape(X_train.shape[0], 40, 32, 1)
X_val = X_val.reshape(X_val.shape[0], 40, 32, 1)
X_test = X_test.reshape(X_test.shape[0], 40, 32, 1)

print(f"Data ready for CNN. Train shape: {X_train.shape}")

Data ready for CNN. Train shape: (7492, 40, 32, 1)


#### Model A:Simple 2D CNN (Baseline)
##### Uses classic 2D convolutions. This is the high-parameter reference point to show why optimization is necessary.

In [ ]:
def build_simple_cnn_for_qat(input_shape, num_classes):
    return tf_keras.Sequential([
        tf_keras.layers.Input(shape=input_shape),
        tf_keras.layers.Conv2D(32, (3,3), activation='relu'),
        tf_keras.layers.MaxPooling2D((2,2)),
        tf_keras.layers.Conv2D(64, (3,3), activation='relu'),
        tf_keras.layers.MaxPooling2D((2,2)),
        tf_keras.layers.Flatten(),
        tf_keras.layers.Dense(64, activation='relu'),
        tf_keras.layers.Dropout(0.3),
        tf_keras.layers.Dense(3, activation='softmax')
    ])

model_a = build_simple_cnn_for_qat((40, 32, 1), 3)
model_a.compile(
    optimizer=tf_keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_a.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_9 (Conv2D)           (None, 38, 30, 32)        320       
                                                                 
 max_pooling2d_5 (MaxPoolin  (None, 19, 15, 32)        0         
 g2D)                                                            
                                                                 
 conv2d_10 (Conv2D)          (None, 17, 13, 64)        18496     
                                                                 
 max_pooling2d_6 (MaxPoolin  (None, 8, 6, 64)          0         
 g2D)                                                            
                                                                 
 flatten_1 (Flatten)         (None, 3072)              0         
                                                                 
 dense_5 (Dense)             (None, 64)               

### Model B:Mini-SqueezeNet
##### A specific lightweight architecture, uses Fire Modules (Squeeze & Expand).
 how 1*1 convolutions can 'squeeze' information to save memory.

In [ ]:
def fire_module(x, squeeze, expand):
    s = layers.Conv2D(squeeze, (1, 1), padding='same', activation='relu')(x)
    e1 = layers.Conv2D(expand, (1, 1), padding='same', activation='relu')(s)
    e3 = layers.Conv2D(expand, (3, 3), padding='same', activation='relu')(s)
    return layers.Concatenate()([e1, e3])

def build_squeezenet(input_shape, num_classes):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(16, (3, 3), strides=(1, 1), padding='same', activation='relu')(inputs)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)

    # The Fire Modules
    x = fire_module(x, squeeze=8, expand=16)
    x = fire_module(x, squeeze=8, expand=16)

    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs, name="Model_B_SqueezeNet")
    return model

# Create and inspect
model_b = build_squeezenet((40, 32, 1), 3)
model_b.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model_b.summary()

Model: "Model_B_SqueezeNet"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_5 (InputLayer)        [(None, 40, 32, 1)]          0         []                            
                                                                                                  
 conv2d_11 (Conv2D)          (None, 40, 32, 16)           160       ['input_5[0][0]']             
                                                                                                  
 max_pooling2d_7 (MaxPoolin  (None, 20, 16, 16)           0         ['conv2d_11[0][0]']           
 g2D)                                                                                             
                                                                                                  
 conv2d_12 (Conv2D)          (None, 20, 16, 8)            136       ['max_pooling

### Model C:Ultra-Lightweight (MobileNet-style)
##### MobileNet-inspired, uses Depthwise Separable Convolutions and Global Average Pooling.

In [ ]:
def build_ultralight_cnn(input_shape, num_classes):
    model = models.Sequential([
        layers.Input(shape=input_shape),

        # Block 1: Feature Extraction
        layers.SeparableConv2D(16, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),

        # Block 2: Feature Extraction
        layers.SeparableConv2D(32, (3, 3), padding='same', activation='relu'),
        layers.MaxPooling2D((2, 2)),

        # The "Embedded Trick": Global Average Pooling instead of Flatten
        layers.GlobalAveragePooling2D(),

        # Final Classification Head
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

# 1. Initialize Model C
model_c = build_ultralight_cnn((40, 32, 1), 3)

# 2. Compile it (This ensures the 'Adam' name is recognized)
model_c.compile(optimizer=Adam(learning_rate=0.001),
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])

# 3. View Summary
model_c.summary()

Model: "sequential_4"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 separable_conv2d_4 (Separa  (None, 40, 32, 16)        41        
 bleConv2D)                                                      
                                                                 
 max_pooling2d_10 (MaxPooli  (None, 20, 16, 16)        0         
 ng2D)                                                           
                                                                 
 separable_conv2d_5 (Separa  (None, 20, 16, 32)        688       
 bleConv2D)                                                      
                                                                 
 max_pooling2d_11 (MaxPooli  (None, 10, 8, 32)         0         
 ng2D)                                                           
                                                                 
 global_average_pooling2d_4  (None, 32)               

##**Traing**
##### I utilized a batch size of 32 and 30 epochs to ensure stable gradient convergence while avoiding overfitting on the phonetically similar classes. I maintained the standard Adam learning rate of 0.001 for the initial training phase.

In [ ]:
models_to_train = {
    "Model_A_Standard": model_a,
    "Model_B_Separable": model_b,
    "Model_C_UltraLight": model_c
}

EPOCHS = 30
BATCH_SIZE = 32
LEARNING_RATE = 0.001

histories = {}
training_times = {}

for name, model in models_to_train.items():
    print(f"\n Starting Training for {name} ")

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    start_train = time.time()

    history = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val, y_val),
        verbose=1
    )
    end_train = time.time()

    training_times[name] = end_train - start_train
    histories[name] = history

    history_df = pd.DataFrame(history.history)
    history_csv = os.path.join(log_path, f"{name}_training_log.csv")
    history_df.to_csv(history_csv, index=False)

    print(f" {name} log successfully saved to: {history_csv}")
    print(f"{name} Training Complete.")



 Starting Training for Model_A_Standard 
Epoch 1/30
235/235 [==============================] - 24s 89ms/step - loss: 0.7550 - accuracy: 0.7090 - val_loss: 0.4182 - val_accuracy: 0.8239
Epoch 2/30
235/235 [==============================] - 14s 58ms/step - loss: 0.3452 - accuracy: 0.8619 - val_loss: 0.2845 - val_accuracy: 0.8773
Epoch 3/30
235/235 [==============================] - 13s 56ms/step - loss: 0.2697 - accuracy: 0.8984 - val_loss: 0.3291 - val_accuracy: 0.8837
Epoch 4/30
235/235 [==============================] - 12s 49ms/step - loss: 0.2223 - accuracy: 0.9132 - val_loss: 0.2618 - val_accuracy: 0.8943
Epoch 5/30
235/235 [==============================] - 13s 54ms/step - loss: 0.1908 - accuracy: 0.9255 - val_loss: 0.2448 - val_accuracy: 0.9072
Epoch 6/30
235/235 [==============================] - 13s 56ms/step - loss: 0.1658 - accuracy: 0.9365 - val_loss: 0.2512 - val_accuracy: 0.9104
Epoch 7/30
235/235 [==============================] - 14s 60ms/step - loss: 0.1558 - accuracy:

### The Quantitative Comparison

In [ ]:
def generate_comparison_table(models_dict, history_dict, training_times_dict):
    stats = []

    for name, model in models_dict.items():

        params = model.count_params()
        # Memory estimation: each param is 4 bytes (float32)
        size_kb = (params * 4) / 1024

        final_val_acc = history_dict[name].history['val_accuracy'][-1]

        _ = model.predict(X_test[:5], verbose=0)

        test_samples = X_test[:100]
        start_inf = time.perf_counter()
        _ = model.predict(test_samples, verbose=0)
        end_inf = time.perf_counter()

        # Calculate ms per sample
        inf_latency_ms = ((end_inf - start_inf) / 100) * 1000

        stats.append({
            "Model Architecture": name,
            "Total Parameters": f"{params:,}",
            "Est. Memory (KB)": round(size_kb, 2),
            "Final Val Acc": f"{final_val_acc:.2%}",
            "Train Time (sec)": round(training_times_dict[name], 2),
            "Inf Latency (ms)": round(inf_latency_ms, 3)
        })
    return pd.DataFrame(stats)

# Generate and print
results_df = generate_comparison_table(models_to_train, histories, training_times)
print("\n Final Project Analysis: DL Models")
print(results_df)

baseline_csv = os.path.join(results_path, 'baseline_model_comparison.csv')
results_df.to_csv(baseline_csv, index=False)


 Final Project Analysis: DL Models
   Model Architecture Total Parameters  Est. Memory (KB) Final Val Acc  \
0    Model_A_Standard          215,683            842.51        90.61%   
1   Model_B_Separable            3,283             12.82        89.75%   
2  Model_C_UltraLight            1,884              7.36        86.02%   

   Train Time (sec)  Inf Latency (ms)  
0            446.26             1.301  
1            350.05             1.069  
2            277.58             1.052  


## **Post-Training Quantization (PTQ)**
##### In this step, we capture the architectural parameters and the conversion time (the "train time" of the optimization process).

In [ ]:
ptq_metadata = []

def representative_data_gen():
    for i in range(100):
        data = np.expand_dims(X_train[i], axis=0).astype(np.float32)
        yield [data]

for name, model in models_to_train.items():
    print(f"\n Processing PTQ: {name}")
    total_params = model.count_params()

    try:
        start_ptq = time.time()

        saved_model_path = os.path.join(results_path, f'{name}_ptq_saved')
        model.export(saved_model_path)

        converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_path)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = representative_data_gen
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.float32
        converter.inference_output_type = tf.float32

        tflite_model = converter.convert()
        ptq_duration = time.time() - start_ptq

        file_path = os.path.join(results_path, f'{name}_ptq.tflite')
        with open(file_path, 'wb') as f:
            f.write(tflite_model)

        ptq_metadata.append({
            "Model": name,
            "Type": "PTQ",
            "Total Parameters": f"{total_params:,}",
            "Train Time (sec)": round(ptq_duration, 2),
            "Size (KB)": round(len(tflite_model) / 1024, 2),
            "path": file_path
        })
        print(f" {name} PTQ saved. Time: {ptq_duration:.2f}s")

    except Exception as e:
        print(f" Error converting {name}: {str(e)}")

pd.DataFrame(ptq_metadata).to_csv(os.path.join(log_path, 'ptq_process_summary.csv'), index=False)


 Processing PTQ: Model_A_Standard
Saved artifact at '/content/drive/MyDrive/speech_commands_project/results/Model_A_Standard_ptq_saved'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 40, 32, 1), dtype=tf.float32, name='input_4')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  135897467628880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135897976652240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135897333059024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135897333056528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135897333057296: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135897333056336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135897333058832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135897333057488: TensorSpec(shape=(), dtype=tf.resource, name=None)
 Model_A_Standard PTQ saved. Time: 2.74s

 Processing PTQ: Mode

## **Quantization-Aware Training (QAT)**

In [ ]:
qat_metadata = []
qat_tflite_files = {}

qat_models = {
    "Model_A_Standard": tfmot.quantization.keras.quantize_model(model_a),
    "Model_B_Separable": tfmot.quantization.keras.quantize_model(model_b),
    "Model_C_UltraLight": tfmot.quantization.keras.quantize_model(model_c)
}

for name, q_model in qat_models.items():
    print(f"\n Starting QAT Fine-tuning: {name}")
    total_params = q_model.count_params()

    q_model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    #  training timer
    start_train = time.time()
    history = q_model.fit(
        X_train, y_train,
        epochs=8,
        batch_size=32,
        validation_data=(X_val, y_val),
        verbose=1
    )
    train_duration = time.time() - start_train

    pd.DataFrame(history.history).to_csv(os.path.join(log_path, f"{name}_qat_history.csv"), index=False)

    # Convert to TFLite
    saved_qat_path = os.path.join(results_path, f'{name}_qat_saved')
    q_model.export(saved_qat_path)
    converter = tf.lite.TFLiteConverter.from_saved_model(saved_qat_path)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_qat_model = converter.convert()

    file_path = os.path.join(results_path, f'{name}_qat.tflite')
    with open(file_path, 'wb') as f:
        f.write(tflite_qat_model)

    qat_metadata.append({
        "Model": name,
        "Type": "QAT",
        "Total Parameters": f"{total_params:,}",
        "Train Time (sec)": round(train_duration, 2),
        "Size (KB)": round(len(tflite_qat_model) / 1024, 2),
        "path": file_path
    })
    print(f" {name} QAT saved. Time: {train_duration:.2f}s")

pd.DataFrame(qat_metadata).to_csv(os.path.join(log_path, 'qat_process_summary.csv'), index=False)


 Starting QAT Fine-tuning: Model_A_Standard
Epoch 1/8
235/235 [==============================] - 35s 139ms/step - loss: 0.2850 - accuracy: 0.9014 - val_loss: 0.6363 - val_accuracy: 0.8495
Epoch 2/8
235/235 [==============================] - 22s 93ms/step - loss: 0.0409 - accuracy: 0.9863 - val_loss: 0.5343 - val_accuracy: 0.8922
Epoch 3/8
235/235 [==============================] - 15s 65ms/step - loss: 0.0191 - accuracy: 0.9923 - val_loss: 0.5388 - val_accuracy: 0.9061
Epoch 4/8
235/235 [==============================] - 15s 64ms/step - loss: 0.0180 - accuracy: 0.9933 - val_loss: 0.5459 - val_accuracy: 0.9050
Epoch 5/8
235/235 [==============================] - 15s 64ms/step - loss: 0.0134 - accuracy: 0.9941 - val_loss: 0.5591 - val_accuracy: 0.9072
Epoch 6/8
235/235 [==============================] - 16s 69ms/step - loss: 0.0149 - accuracy: 0.9947 - val_loss: 0.5605 - val_accuracy: 0.9082
Epoch 7/8
235/235 [==============================] - 22s 93ms/step - loss: 0.0111 - accuracy: 0.

## **Evaluation**

In [ ]:
def evaluate_tflite_and_measure(file_path, X_test, y_test):

    interpreter = tf.lite.Interpreter(model_path=file_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    correct = 0
    start_eval = time.perf_counter()

    for i in range(len(X_test)):
        # Ensure data is float32
        input_data = X_test[i:i+1].astype(np.float32)
        interpreter.set_tensor(input_details['index'], input_data)
        interpreter.invoke()
        output_data = interpreter.get_tensor(output_details['index'])

        if np.argmax(output_data) == y_test[i]:
            correct += 1

    eval_duration = time.perf_counter() - start_eval

    avg_latency_ms = (eval_duration / len(X_test)) * 1000
    accuracy = correct / len(X_test)
    size_kb = os.path.getsize(file_path) / 1024

    return accuracy, size_kb, avg_latency_ms

final_benchmark = []

if 'ptq_metadata' in globals() and ptq_metadata:
    for entry in ptq_metadata:
        acc, size, lat = evaluate_tflite_and_measure(entry["path"], X_test, y_test)

        final_benchmark.append({
            "Model": entry["Model"],
            "Type": "PTQ",
            "Total Parameters": entry["Total Parameters"],
            "Final Val Acc": f"{acc:.2%}",
            "Est. Memory (KB)": round(size, 2),
            "Train Time (sec)": entry["Train Time (sec)"],
            "Inf Latency (ms)": round(lat, 2)
        })

if 'qat_metadata' in globals() and qat_metadata:
    for entry in qat_metadata:
        acc, size, lat = evaluate_tflite_and_measure(entry["path"], X_test, y_test)

        final_benchmark.append({
            "Model": entry["Model"],
            "Type": "QAT",
            "Total Parameters": entry["Total Parameters"],
            "Final Val Acc": f"{acc:.2%}",
            "Est. Memory (KB)": round(size, 2),
            "Train Time (sec)": entry["Train Time (sec)"],
            "Inf Latency (ms)": round(lat, 2)
        })

results_df = pd.DataFrame(final_benchmark)
final_csv_path = os.path.join(results_path, 'ptq_qat_analysis.csv')
results_df.to_csv(final_csv_path, index=False)

print("\n FINAL COMPRESSION EXPERIMENT ANALYSIS:")
display(results_df)

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



 FINAL COMPRESSION EXPERIMENT ANALYSIS:


,Model,Type,Total Parameters,Final Val Acc,Est. Memory (KB),Train Time (sec),Inf Latency (ms)
0,Model_A_Standard,PTQ,"215,683",93.28%,219.02,2.74,0.62
1,Model_B_Separable,PTQ,"3,283",92.32%,12.22,1.96,0.37
2,Model_C_UltraLight,PTQ,"1,884",84.31%,9.66,1.68,0.17
3,Model_A_Standard,QAT,"215,898",93.28%,217.93,156.40,0.71
4,Model_B_Separable,QAT,"3,514",92.64%,13.27,144.61,0.48
5,Model_C_UltraLight,QAT,"2,015",82.60%,9.48,112.78,0.16


## Merge Analysis

In [ ]:
baseline_csv = os.path.join(results_path, 'baseline_model_comparison.csv')
final_analysis_csv = os.path.join(results_path, 'ptq_qat_analysis.csv')

if os.path.exists(baseline_csv) and os.path.exists(final_analysis_csv):

    df_base = pd.read_csv(baseline_csv)
    df_opt = pd.read_csv(final_analysis_csv)

    df_base_clean = df_base.rename(columns={
        'Model Architecture': 'Model',
        'Final Val Acc': 'Acc',
        'Est. Memory (KB)': 'Size (KB)',
        'Inf Latency (ms)': 'Latency (ms)'
    })
    df_base_clean['Type'] = 'Baseline'

    df_opt_clean = df_opt.rename(columns={
        'Final Val Acc': 'Acc',
        'Est. Memory (KB)': 'Size (KB)',
        'Inf Latency (ms)': 'Latency (ms)'
    })

    df_final = pd.concat([df_base_clean, df_opt_clean], ignore_index=True)

    def clean_val(x):
        if isinstance(x, str):
            return float(x.replace('%', '').replace(',', ''))
        return x

    df_final['Size (KB)'] = df_final['Size (KB)'].apply(clean_val).round(2)
    df_final['Latency (ms)'] = df_final['Latency (ms)'].apply(clean_val).round(2)
    df_final['Train Time (sec)'] = df_final['Train Time (sec)'].apply(clean_val).round(2)

    def to_percent(x):
        val = clean_val(x)
        if val <= 1.0: val *= 100
        return f"{val:.2f}%"

    df_final['Acc'] = df_final['Acc'].apply(to_percent)

    cols = ['Model', 'Type', 'Total Parameters', 'Acc', 'Size (KB)', 'Train Time (sec)', 'Latency (ms)']
    df_final = df_final[cols]

    final_csv_path = os.path.join(results_path, 'final_result.csv')
    df_final.to_csv(final_csv_path, index=False)

    print(f" Master Result saved to: {final_csv_path}")
    display(df_final)
else:
    print(" Error: Missing baseline_model_comparison.csv or final_analysis.csv")

 Master Result saved to: /content/drive/MyDrive/speech_commands_project/results/final_result.csv


,Model,Type,Total Parameters,Acc,Size (KB),Train Time (sec),Latency (ms)
0,Model_A_Standard,Baseline,"215,683",90.61%,842.51,446.26,1.30
1,Model_B_Separable,Baseline,"3,283",89.75%,12.82,350.05,1.07
2,Model_C_UltraLight,Baseline,"1,884",86.02%,7.36,277.58,1.05
3,Model_A_Standard,PTQ,"215,683",93.28%,219.02,2.74,0.62
4,Model_B_Separable,PTQ,"3,283",92.32%,12.22,1.96,0.37
5,Model_C_UltraLight,PTQ,"1,884",84.31%,9.66,1.68,0.17
6,Model_A_Standard,QAT,"215,898",93.28%,217.93,156.40,0.71
7,Model_B_Separable,QAT,"3,514",92.64%,13.27,144.61,0.48
8,Model_C_UltraLight,QAT,"2,015",82.60%,9.48,112.78,0.16
